## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models import convnext_base, ConvNeXt_Base_Weights, resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

from data.datasets import DeepFakeDataset
from torch.utils.tensorboard import SummaryWriter

from torcheval.metrics.functional import binary_auroc, binary_auprc

In [ ]:
image_dir_path = 'Deepfake-Eval-2024/image-data'

train_data = DeepFakeDataset("image-metadata-train.csv", image_dir_path, 'ResNet', is_train = True)
train_data_loader = DataLoader(train_data, batch_size = 32, shuffle = True)

val_data = DeepFakeDataset("image-metadata-val.csv", image_dir_path, 'ResNet', is_train = True)
val_data_loader = DataLoader(val_data, batch_size = 32, shuffle = True)

# small subset of training data to run on CPU and debug functional issues
debug_data = DeepFakeDataset("image-metadata-debug.csv", image_dir_path, 'ResNet', is_train = True)
debug_data_loader = DataLoader(debug_data, batch_size = 32, shuffle = False)

# model = convnext_base(weights = ConvNeXt_Base_Weights.DEFAULT)
model = resnet18(weights = ResNet18_Weights.DEFAULT)


In [ ]:
# count = 0
# features, label = next(iter(debug_data_loader))
# with torch.no_grad():
#     model_output = model(features)
#     print(model_output.shape)

# for i in range(5):

#     img = features[i]
#     # print(predictions[0].argmax(-1).item())
#     imagenet_label = ConvNeXt_Base_Weights.DEFAULT.meta['categories'][model_output[i].argmax(-1).item()]

#     plt.imshow(np.transpose(img, (1,2,0))) # convert pytorch tensor (3 channels, H, W) to numpy array (H, W, 3 channels)
#     plt.title(f"Ground Truth Label: {"Fake" if label[i] else "Real"}\nImageNet Label: {imagenet_label}")
#     plt.xticks([])
#     plt.yticks([])
#     plt.show()

In [ ]:
# # modify ConvNeXtclassifier head for deepfake detection
# classifier_head_layers = [layer for layer in model.classifier]
# classifier_head_layers[2] = nn.Linear(in_features=1024, out_features=2, bias = True)
# model.classifier = nn.Sequential(*classifier_head_layers)

In [ ]:
# modify ResNet classifier head for deepfake detection
model.fc = nn.Linear(in_features = 512, out_features=2, bias = True)

In [ ]:
# # run this for classification models from torch.hub
# # replace classification head with passthrough effectively removing last classification layer
# model.fc = nn.Identity()    
# # extract features for train
# resnet_features = model(features)

In [ ]:
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# model.to(device)
learning_rate = 1e-3
batch_size = 512
epochs = 10
loss_fn = nn.CrossEntropyLoss(reduction = 'sum')
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
# optimizer = torch.optim.LBFGS(model.parameters(), max_iter = 5, history_size = 10)

In [ ]:
def train_epoch(dataloader, model, loss_fn, optimizer):
    # learn the weights
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    
    # # evaluate performance on training data
    # model.eval()
    # train_loss = 0
    # all_preds = torch.zeros((0,2))
    # all_labels = torch.zeros(0)
    
    # with torch.no_grad():
    #     for batch, (X, y) in enumerate(dataloader):
    #         pred = model(X)
    #         all_preds = torch.cat((all_preds, pred), dim = 0)
    #         all_labels = torch.cat((all_labels, y), dim = 0)
    #         train_loss += loss_fn(pred, y).item()
    
    # train_auroc = binary_auroc(all_preds[:,1], all_labels)
    # train_auprc = binary_auprc(all_preds[:,1], all_labels)
    # train_loss /= len(dataloader.dataset)

    # train_auroc, train_auprc, train_loss = validate_epoch(debug_data_loader, model, loss_fn)

    # print(f"Training Error: \n\tLoss: {train_loss:>8f}\tROC AUC: {train_auroc:>4f}\tPR AUC: {train_auprc:>4f} \n")
    # print(f"Validation ROC AUC: {val_roc_auc}; Validation PR AUC: {val_roc_auc} \n")

    # evaluate and return performance on training data (only after iterating through all data)
    return validate_epoch(dataloader, model, loss_fn)

def validate_epoch(dataloader, model, loss_fn):
    model.eval()
    val_loss = 0
    all_preds = torch.zeros((0,2))
    all_labels = torch.zeros(0)
    
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            all_preds = torch.cat((all_preds, pred), dim = 0)
            all_labels = torch.cat((all_labels, y), dim = 0)
            val_loss += loss_fn(pred, y).item()


    val_auroc = binary_auroc(all_preds[:,1], all_labels)
    val_auprc = binary_auprc(all_preds[:,1], all_labels)
    val_loss /= len(dataloader.dataset)
    # print(f"Validation Error: \n\tLoss: {val_loss:>8f}\tROC AUC: {val_auroc:>4f}\tPR AUC: {val_auprc:>4f} \n")
    # print(f"Validation ROC AUC: {val_roc_auc}; Validation PR AUC: {val_roc_auc} \n")

    return val_loss, val_auroc, val_auprc

In [ ]:
# writer = SummaryWriter()
train_loss_, train_auroc_, train_auprc_, val_loss_, val_auroc_, val_auprc_ = [None] * epochs, [None] * epochs, [None] * epochs, [None] * epochs, [None] * epochs, [None] * epochs
for t in range(epochs):
    print(f"Epoch {t+1}\n- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - ")
    train_loss, train_auroc, train_auprc = train_epoch(debug_data_loader, model, loss_fn, optimizer)
    val_loss, val_auroc, val_auprc = validate_epoch(debug_data_loader, model, loss_fn)
#     writer.add_scalar('training loss', train_loss, t)
#     writer.add_scalar('validation loss', val_loss, t)
#     writer.add_scalar('training roc auc', train_auroc, t)
#     writer.add_scalar('validation roc auc', val_auroc, t)
#     writer.add_scalar('training pr auc', train_auprc, t)
#     writer.add_scalar('validation pr auc', val_auprc, t)
    print(f"Training Error: \n\tLoss: {train_loss:>8f}\tROC AUC: {train_auroc:>4f}\tPR AUC: {train_auprc:>4f} \n")
    print(f"Validation Error: \n\tLoss: {val_loss:>8f}\tROC AUC: {val_auroc:>4f}\tPR AUC: {val_auprc:>4f} \n")
    train_loss_[t], train_auroc_[t], train_auprc_[t], val_loss_[t], val_auroc_[t], val_auprc_[t] = train_loss, train_auroc, train_auprc, val_loss, val_auroc, val_auprc
    # train_loss_.append(train_loss)
    # train_auroc_.append(train_auroc)
    # train_auprc_.append(train_auprc)
    # val_loss_.append(val_loss)
    # val_auroc_.append(val_auroc)
    # val_auprc_.append(val_auprc)
# writer.close()
print("Done!")

In [ ]:


with open(f"train_progress.pkl", "wb") as f:
    pickle.dump((train_loss_, train_auroc_, train_auprc_, val_loss_, val_auroc_, val_auprc_), f)
    f.close()

In [ ]:
import pickle
with open("train_progress.pkl", "rb") as f:
    train_loss_, train_auroc_, train_auprc_, val_loss_, val_auroc_, val_auprc_ = pickle.load(f)

In [ ]:
# plt.rcParams['font.family'] = 'serif'
# plt.rcParams['font.serif'] = ['Times New Roman']
# plt.rcParams['font.size'] = 10
# plt.rcParams["axes.formatter.use_mathtext"] = True
# plt.rcParams["text.usetex"] = True

In [ ]:
plt.plot(train_loss_, label = 'Training Loss')
plt.plot(val_loss_, label = 'Validation Loss')
plt.grid(axis = 'both')
plt.legend()
plt.show()